# b02 허브 다시 찍기 · 무릎 통과와 바 접촉

판독할 곳은 카메라에 가까운 원판을 바벨 축이 관통하는 중심 한 점입니다. 각 탭에서 원본을 보고 한 번 클릭하면 저장됩니다. 필요하면 원본 크기로 확대하세요. 한 점을 자신 있게 정할 수 없으면 **판독 불가로 저장**하세요. 이 화면에는 이번에 내가 찍은 점만 표시됩니다.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from wlcv.annotation.hub_points import hub_editor, read_hubs
from wlcv.evaluation.hub_rise_repeat import FRAMES, POINTS

tabs = widgets.Tab(children=[hub_editor(i, path=POINTS, frames=FRAMES) for i in FRAMES])
for tab, frame in enumerate(FRAMES):
    tabs.set_title(tab, f"{frame}번")
display(tabs)

## 두 탭을 모두 저장한 뒤 실행

남은 프레임이 있으면 아래 셀에 번호가 표시됩니다. 두 점을 직접 찍으면 1차와 2차 판독의 상승량, 두 모델 입력 크기의 오차와 판정이 표시됩니다. 출력 확인 후에는 이 두 점이 블라인드 판독 기록이므로 그대로 보존하세요.

In [ ]:
import pandas as pd
from wlcv.config import ROOT
from wlcv.evaluation.hub_rise_repeat import compare_repeat

repeat = read_hubs(POINTS, frames=FRAMES)
print(f"{len(repeat)}/2 저장")
remaining = sorted(set(FRAMES) - set(repeat.frame_index))
if remaining:
    print("판독할 프레임:", remaining)
elif not repeat.visibility.eq("visible").all():
    print("판독 불가로 저장된 프레임이 있어 이 구간의 상승량은 비교할 수 없습니다.")
else:
    # 37번 실험에서 보관한 결과를 사용해 같은 모델 출력에 두 판독을 대조합니다.
    baseline_path = ROOT / "outputs/phase_hub_rise/20261124-194908-305541/phase_rises.csv"
    points, results = compare_repeat(pd.read_csv(baseline_path), repeat)
    display(points, results.round(2))
    first, second = results.to_dict("records")
    same = (first["두 입력 모두 과소측정"] == second["두 입력 모두 과소측정"]
            and first["절대 오차가 작은 입력"] == second["절대 오차가 작은 입력"])
    print("과소측정 여부와 절대 오차 순위:", "두 판독에서 유지" if same else "재판독에서 달라짐")
    output = ROOT / "outputs/b02_hub_reread"
    output.mkdir(parents=True, exist_ok=True)
    points.to_csv(output / "endpoint_changes.csv", index=False)
    results.to_csv(output / "rise_comparison.csv", index=False)